# CAPM（资本资产定价模型）— 学习笔记

本 notebook 是 [马科维兹理论.ipynb](马科维兹理论.ipynb) 的续篇：马科维兹回答「给定 μ、Σ，最优权重怎么配」；CAPM 回答「单只股票的风险溢价如何由市场系统性风险 β 决定」。

**学习路径**：合成截面 CAPM → 合成时间序列 OLS → 真实 A 股回归估 β → SML 图

---

## 一、理论公式 ↔ 代码变量

| 概念 | 公式 | 代码变量 |
|------|------|----------|
| 超额收益 | $R_i - r_f$ | `excess_i`, `excess_m` |
| 理论 Beta | $\beta_i = \text{Cov}(R_i,R_m)/\text{Var}(R_m)$ | `beta_theory` |
| CAPM 预期收益 | $E[R_i] = r_f + \beta_i(E[R_m]-r_f)$ | `capm_mu` |
| 回归形式 | $R_i-r_f = \alpha + \beta(R_m-r_f)+\varepsilon$ | `sm.OLS` |
| 证券市场线 SML | 横轴 β，纵轴 $E[R_i]$ | `plt.plot(beta_line, sml_line)` |

---

## 二、与马科维兹的衔接

| | 马科维兹 CML | CAPM SML |
|--|-------------|----------|
| 横轴 | 组合标准差 σ | Beta β |
| 适用对象 | **有效组合**（前沿上） | **任意资产**（含非有效组合） |
| 核心思想 | 分散化降低非系统性风险 | 只有系统性风险（β）获得补偿 |

> **读图要点**：SML 上每一点满足「收益 = 无风险利率 + β × 市场风险溢价」；回归截距 α（Jensen's α）显著不为 0 时，说明实际收益偏离 CAPM 预测。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import warnings

warnings.filterwarnings('ignore')
plt.rcParams['font.sans-serif'] = ['SimHei']
plt.rcParams['axes.unicode_minus'] = False

# ===================== Part 1：合成截面 CAPM =====================
# 复用马科维兹笔记的 4 资产参数，保持学习连贯性
asset_labels = ['股票A', '股票B', '债券', '黄金']
mus = np.array([0.12, 0.09, 0.04, 0.07])       # 年化预期收益率
sigmas = np.array([0.25, 0.18, 0.06, 0.15])    # 年化标准差
rho_matrix = np.array([
    [1.00, 0.70, 0.20, 0.30],
    [0.70, 1.00, 0.10, 0.20],
    [0.20, 0.10, 1.00, 0.05],
    [0.30, 0.20, 0.05, 1.00],
])
cov_matrix = np.outer(sigmas, sigmas) * rho_matrix

rf = 0.03  # 无风险利率 3%（马科维兹笔记用 0，这里更贴近真实）

# 市场组合：等权（学习用；真实市场常用市值加权）
w_m = np.ones(4) / 4
mu_m = w_m @ mus
sigma_m = np.sqrt(w_m @ cov_matrix @ w_m)

# 理论 Beta：β_i = Cov(R_i, R_m) / Var(R_m)
# 其中 Cov(R_i, R_m) = (Σw)_i 在组合权重为 w_m 时
cov_with_market = cov_matrix @ w_m
var_market = w_m @ cov_matrix @ w_m
beta_theory = cov_with_market / var_market

# CAPM 定价：E[R_i] = r_f + β_i * (E[R_m] - r_f)
market_premium = mu_m - rf
capm_mu = rf + beta_theory * market_premium

# ===================== 验证与输出 =====================
print('=== Part 1：合成截面 CAPM ===')
print(f'市场组合 E[R_m]={mu_m:.4f}, σ_m={sigma_m:.4f}')
print(f'市场风险溢价 E[R_m]-r_f={market_premium:.4f}')
print('\n理论 Beta 与 CAPM 定价收益：')
for i, name in enumerate(asset_labels):
    print(f'  {name}: β={beta_theory[i]:.4f}, CAPM μ={capm_mu[i]:.4f}, 输入 μ={mus[i]:.4f}')

# 输入 μ 来自马科维兹例子，未必满足 CAPM 均衡；打印偏差帮助理解
max_dev = np.max(np.abs(capm_mu - mus))
print(f'\n|CAPM μ - 输入 μ| 最大偏差: {max_dev:.4f}')
print('（偏差>0 说明任意给定的 μ 不一定处于 CAPM 均衡；SML 图用 CAPM μ 作纵轴）')

# ===================== SML 图 =====================
beta_line = np.linspace(0, beta_theory.max() * 1.2, 100)
sml_line = rf + beta_line * market_premium

plt.figure(figsize=(10, 6))
plt.plot(beta_line, sml_line, 'k--', linewidth=2, label='证券市场线 SML')
plt.scatter(beta_theory, capm_mu, s=150, c='steelblue', edgecolors='black', zorder=5, label='CAPM 定价点')
plt.scatter(beta_theory, mus, s=80, c='coral', marker='x', linewidths=2, zorder=5, label='马科维兹输入 μ')
for i, name in enumerate(asset_labels):
    plt.annotate(name, (beta_theory[i], capm_mu[i]), textcoords='offset points', xytext=(5, 5), fontsize=9)
plt.xlabel('Beta (β)', fontsize=12)
plt.ylabel('预期年化收益率', fontsize=12)
plt.title('证券市场线 SML（合成 4 资产）', fontsize=14)
plt.legend(loc='upper left')
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
import warnings

warnings.filterwarnings('ignore')
plt.rcParams['font.sans-serif'] = ['SimHei']
plt.rcParams['axes.unicode_minus'] = False

# ===================== Part 2：合成时间序列 + OLS 回归 =====================
# 从多元正态分布模拟日收益，用 OLS 估 β，与 Part 1 理论值对比
np.random.seed(42)
T = 5000  # 样本期数（日频）；样本越大 OLS β 越接近理论值
n_assets = 4
rf_daily = rf / 252

# 日化均值与协方差（假设 iid 日收益）
mu_daily = mus / 252
cov_daily = cov_matrix / 252

returns = np.random.multivariate_normal(mu_daily, cov_daily, size=T)
market_ret = returns @ w_m

# 超额收益
excess_m = market_ret - rf_daily
excess_assets = returns - rf_daily

beta_ols_list = []
alpha_ols_list = []
for i in range(n_assets):
    X = sm.add_constant(excess_m)
    model = sm.OLS(excess_assets[:, i], X).fit()
    alpha_ols_list.append(model.params[0])
    beta_ols_list.append(model.params[1])

beta_ols = np.array(beta_ols_list)
alpha_ols = np.array(alpha_ols_list)

# ===================== 对比表 =====================
compare_df = pd.DataFrame({
    '资产': asset_labels,
    '理论β': beta_theory,
    'OLSβ': beta_ols,
    '|Δβ|': np.abs(beta_theory - beta_ols),
    'OLSα(日化)': alpha_ols,
})
print('=== Part 2：理论 β vs OLS 回归 β ===')
print(compare_df.to_string(index=False, float_format=lambda x: f'{x:.6f}'))
print(f'\n最大 |Δβ| = {compare_df["|Δβ|"].max():.2e}')

# ===================== 单资产回归散点图（股票A） =====================
i_show = 0
X = sm.add_constant(excess_m)
model_show = sm.OLS(excess_assets[:, i_show], X).fit()
fit_line = model_show.predict(X)

plt.figure(figsize=(9, 6))
plt.scatter(excess_m, excess_assets[:, i_show], alpha=0.3, s=10, label='模拟日超额收益')
plt.plot(excess_m, fit_line, 'r-', linewidth=2,
         label=f'OLS 拟合: α={model_show.params[0]:.2e}, β={model_show.params[1]:.4f}')
plt.xlabel('市场超额收益 (日化)', fontsize=12)
plt.ylabel(f'{asset_labels[i_show]} 超额收益 (日化)', fontsize=12)
plt.title(f'CAPM 回归：{asset_labels[i_show]} vs 市场', fontsize=14)
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
import sys
import os
from datetime import date

import numpy as np
import pandas as pd
import polars as pl
import warnings

warnings.filterwarnings('ignore')

# 将项目根目录加入路径（兼容 notebook 从不同目录启动）
PROJECT_ROOT = None
for _candidate in [os.getcwd(), os.path.join(os.getcwd(), '..'), os.path.join(os.getcwd(), '../..')]:
    _root = os.path.abspath(_candidate)
    if os.path.isdir(os.path.join(_root, 'my_utils')):
        PROJECT_ROOT = _root
        break
if PROJECT_ROOT is None:
    raise FileNotFoundError('未找到项目根目录（需包含 my_utils 文件夹）')
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from my_utils.fun import read_day_data

# ===================== Part 3：真实 A 股数据准备 =====================
start_date = date(2022, 1, 1)
end_date = date(2025, 12, 31)
rf_annual = 0.02  # 年化无风险利率 2%（学习用常数；实盘可换国债收益率）
rf_weekly = rf_annual / 52

# 大盘蓝筹（掘金 GM 代码格式），固定列表避免全市场扫描
stock_list = [
    'SHSE.600519',  # 贵州茅台
    'SHSE.600036',  # 招商银行
    'SHSE.601318',  # 中国平安
    'SZSE.300750',  # 宁德时代（创业板属深交所）
    'SHSE.601398',  # 工商银行
    'SHSE.600028',  # 中国石化
    'SHSE.601857',  # 中国石油
]
stock_names = {
    'SHSE.600519': '贵州茅台',
    'SHSE.600036': '招商银行',
    'SHSE.601318': '中国平安',
    'SZSE.300750': '宁德时代',
    'SHSE.601398': '工商银行',
    'SHSE.600028': '中国石化',
    'SHSE.601857': '中国石油',
}

print('=== Part 3：读取真实 A 股日线 ===')
stock_pl = read_day_data(
    start_date, end_date,
    stock_list=stock_list,
    fields=['code', 'trading_date', 'close', 'pre_close', 'name'],
)

# 日收益率：优先用 pre_close，缺失时退化为 pct_change
stock_pl = stock_pl.sort(['code', 'trading_date']).with_columns([
    pl.when(pl.col('pre_close').is_not_null() & (pl.col('pre_close') != 0))
    .then(pl.col('close') / pl.col('pre_close') - 1)
    .otherwise(pl.col('close').pct_change().over('code'))
    .alias('daily_ret'),
])
stock_pd = stock_pl.to_pandas()
stock_pd['trading_date'] = pd.to_datetime(stock_pd['trading_date'])

# 转为周收益：每周取最后一个交易日的收盘价计算周收益率
weekly_stock = (
    stock_pd.set_index('trading_date')
    .groupby('code')['daily_ret']
    .resample('W')
    .apply(lambda x: (1 + x).prod() - 1 if len(x) > 0 else np.nan)
    .reset_index()
    .rename(columns={'daily_ret': 'weekly_ret'})
)
weekly_stock = weekly_stock.dropna(subset=['weekly_ret'])

# 获取沪深300指数作为市场基准（延迟导入，避免无 gm 环境时 Part 1/2 也无法运行）
market_source = '沪深300 (SHSE.000300)'
try:
    from my_utils.stock_api import stock_api
    api = stock_api()
    index_data = api.gm_get_index_day_data(
        index_code='SHSE.000300',
        start_date=start_date.strftime('%Y-%m-%d'),
        end_date=end_date.strftime('%Y-%m-%d'),
    )
    index_pd = index_data.copy()
    index_pd['trading_date'] = pd.to_datetime(index_pd['trading_date'])
    index_pd['daily_ret'] = index_pd['close'].pct_change()
    weekly_market = (
        index_pd.set_index('trading_date')['daily_ret']
        .resample('W')
        .apply(lambda x: (1 + x.dropna()).prod() - 1 if x.notna().any() else np.nan)
        .reset_index()
        .rename(columns={'daily_ret': 'weekly_ret'})
    )
    weekly_market = weekly_market.dropna(subset=['weekly_ret'])
    print(f'市场基准: {market_source}, 周数={len(weekly_market)}')
except Exception as e:
    # 掘金 API 不可用时：用所选股票等权组合作为伪市场
    market_source = '等权股票组合（降级）'
    print(f'指数 API 失败 ({e})，降级为等权市场组合')
    pivot = weekly_stock.pivot(index='trading_date', columns='code', values='weekly_ret')
    weekly_market = pd.DataFrame({
        'trading_date': pivot.index,
        'weekly_ret': pivot.mean(axis=1),
    }).reset_index(drop=True)

# 对齐日期：仅保留股票与市场都有数据的周
weekly_stock = weekly_stock.merge(
    weekly_market[['trading_date', 'weekly_ret']].rename(columns={'weekly_ret': 'market_weekly_ret'}),
    on='trading_date',
    how='inner',
)
weekly_stock['excess_ret'] = weekly_stock['weekly_ret'] - rf_weekly
weekly_stock['excess_market'] = weekly_stock['market_weekly_ret'] - rf_weekly

print(f'股票数={weekly_stock["code"].nunique()}, 对齐后周数={weekly_stock["trading_date"].nunique()}')
print(f'市场数据来源: {market_source}')
weekly_stock.head()

In [ ]:
import statsmodels.api as sm
import pandas as pd
import numpy as np

# ===================== Part 4：真实数据 OLS 估 Beta =====================
results = []
for code, grp in weekly_stock.groupby('code'):
    grp = grp.dropna(subset=['excess_ret', 'excess_market'])
    if len(grp) < 20:
        continue
    y = grp['excess_ret'].values
    X = sm.add_constant(grp['excess_market'].values)
    model = sm.OLS(y, X).fit()

    # 年化平均收益：周收益均值 × 52
    realized_mu = grp['weekly_ret'].mean() * 52

    results.append({
        'code': code,
        'name': stock_names.get(code, code),
        'beta_ols': model.params[1],
        'alpha_ols': model.params[0],          # 周频超额收益截距
        'alpha_annual': model.params[0] * 52,  # 年化 Jensen's α（近似）
        'r_squared': model.rsquared,
        'pvalue_beta': model.pvalues[1],
        'pvalue_alpha': model.pvalues[0],
        'realized_mu': realized_mu,
        'n_weeks': len(grp),
    })

capm_results = pd.DataFrame(results)
print('=== Part 4：真实 A 股 CAPM 回归结果 ===')
print(capm_results.to_string(index=False, float_format=lambda x: f'{x:.4f}'))
print(f'\n成功估计 β 的股票数: {len(capm_results)}')
capm_results

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

plt.rcParams['font.sans-serif'] = ['SimHei']
plt.rcParams['axes.unicode_minus'] = False

# ===================== Part 5：SML 可视化 =====================
market_premium_weekly = weekly_stock['excess_market'].mean()
market_premium_annual = market_premium_weekly * 52

beta_range = np.linspace(0, capm_results['beta_ols'].max() * 1.2, 100)
sml_real = rf_annual + beta_range * market_premium_annual

fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(beta_range, sml_real, 'k--', linewidth=2, label=f'理论 SML (r_f={rf_annual:.0%})')
ax.scatter(
    capm_results['beta_ols'], capm_results['realized_mu'],
    s=120, c='steelblue', edgecolors='black', zorder=5,
)
for _, row in capm_results.iterrows():
    ax.annotate(
        row['name'],
        (row['beta_ols'], row['realized_mu']),
        textcoords='offset points', xytext=(5, 5), fontsize=9,
    )
ax.set_xlabel('OLS Beta (β)', fontsize=12)
ax.set_ylabel('年化平均收益率', fontsize=12)
ax.set_title(f'证券市场线 SML（真实 A 股，市场={market_source}）', fontsize=14)
ax.legend(loc='upper left')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# ===================== Jensen's α 解读 =====================
print('=== Part 5：Jensen\'s α 解读 ===')
print(f'样本期年化市场风险溢价 ≈ {market_premium_annual:.2%}')
for _, row in capm_results.iterrows():
    beta_tag = 'β>1 波动大于市场' if row['beta_ols'] > 1 else 'β<1 波动小于市场'
    alpha_tag = 'α显著为正' if row['pvalue_alpha'] < 0.05 and row['alpha_annual'] > 0 else (
        'α显著为负' if row['pvalue_alpha'] < 0.05 and row['alpha_annual'] < 0 else 'α不显著'
    )
    print(f"  {row['name']}: β={row['beta_ols']:.2f} ({beta_tag}), {alpha_tag}, R²={row['r_squared']:.2f}")

## 三、学习小结

1. **Part 1（合成截面）**：给定协方差矩阵，可用 $\beta_i=\text{Cov}(R_i,R_m)/\text{Var}(R_m)$ 直接算理论 β；CAPM 定价点必然落在 SML 上，但马科维兹笔记里任意给定的 μ 不一定满足 CAPM 均衡。
2. **Part 2（合成时序）**：大样本 OLS 回归的 β 收敛于理论值；α 应接近 0（日化尺度下很小）。
3. **Part 3–5（真实 A 股）**：β>1 表示波动大于市场；Jensen's α 显著说明收益偏离 CAPM 预测（可能来自个股特质风险、因子暴露等）。
4. **与马科维兹的联系**：马科维兹告诉我们「组合」如何分散非系统性风险；CAPM 进一步说，均衡时投资者只承担系统性风险（β），并据此获得风险溢价。